In [7]:
from hyperopt import hp
search_space = {'x': hp.quniform('x', -10, 10, 1), 'y': hp.quniform('y', -15, 15, 1)}
# 1은 건너뛰는 범위(-10, -9, -8, ... 10)
# 물론 -10부터 순서대로 탐색하진 않음
# search_space는 목적함수의 인자로 들어감

In [5]:
search_space

{'x': <hyperopt.pyll.base.Apply at 0x126fc3650>,
 'y': <hyperopt.pyll.base.Apply at 0x126fc1670>}

---
# 목적함수 정의
- 지금은 예시를 위해 목적함수를 어거지로 만들어준거고, 실제로는 블랙박스이다.
- 블랙박스가 반환하는 값은 그 때의 평가 값(loss든, accuracy든, f1이든 .. 등)

In [8]:
from hyperopt import STATUS_OK

def objective_function(search_space):
    x = search_space['x']
    y = search_space['y']
    retval = x**2 - 20*y
    
    return retval # return {'loss': retval, 'status':STATUS_OK}


---
# 정의한 목적함수를 기반으로 최적의 하이퍼 파라미터 값 찾아보기
- 정의한 목적함수에 따라, 최적의 값(목적함수가 최소값을 반환하는 하이퍼 파라미터의 값)은 x=0, y=15이다.
- max_evals값을 더 크게하면 점점 근접한 값을 뱉어낸다.
- rstate값 안넣으면 시드 고정이 안되긴하는데, 경험상 안넣을 때 최적화가 더 잘되는 느낌(?)

In [12]:
from hyperopt import fmin, tpe, Trials
import numpy as np

trial_val = Trials()

best_01 = fmin(fn=objective_function, space=search_space, algo=tpe.suggest, max_evals=5,
               trials=trial_val, rstate=np.random.default_rng(seed=0))

print("Bets Parameter Pair: ", best_01)

100%|██████████| 5/5 [00:00<00:00, 2211.49trial/s, best loss: -224.0]
Bets Parameter Pair:  {'x': np.float64(-4.0), 'y': np.float64(12.0)}


### trial_val은 뭘까
- 주요한 속성으로 `.results`, `.vals`를 가지고 있는데, 각각 찍어보면 다음과 같다.

In [17]:
print(trial_val.results)
print(trial_val.vals)

[{'loss': -64.0, 'status': 'ok'}, {'loss': -184.0, 'status': 'ok'}, {'loss': 56.0, 'status': 'ok'}, {'loss': -224.0, 'status': 'ok'}, {'loss': 61.0, 'status': 'ok'}]
{'x': [np.float64(-6.0), np.float64(-4.0), np.float64(4.0), np.float64(-4.0), np.float64(9.0)], 'y': [np.float64(5.0), np.float64(10.0), np.float64(-2.0), np.float64(12.0), np.float64(1.0)]}


### trial_val을 이용해서 히스토리 테이블 만들어보기

In [18]:
import pandas as pd

losses = [loss_dict['loss'] for loss_dict in trial_val.results]

result_df = pd.DataFrame({'x': trial_val.vals['x'],
                          'y': trial_val.vals['y'],
                          'losses': losses})

result_df

,x,y,losses
0,-6.0,5.0,-64.0
1,-4.0,10.0,-184.0
2,4.0,-2.0,56.0
3,-4.0,12.0,-224.0
4,9.0,1.0,61.0
